# Catastrophe Risk Modeling: Cyclone Fani (2019) Coastal Odisha Portfolio
### ETH Zurich CLIMADA & Oasis LMF Open Framework Implementation

This interactive notebook demonstrates the complete catastrophe risk modeling lifecycle:
1. **Exposure Scrubbing**: Ingestion of ~3,500 locations across coastal Odisha, SQL cleansing in SQLite, standardizing construction and occupancy codes.
2. **Spatial Geocoding**: Address resolution and coordinate enrichment with `geopy`, calculating distance to coastline.
3. **Hazard Footprint (NOAA IBTrACS)**: Historical track of Cyclone Fani (May 2019, Cat 4/5 equivalent, landfall at Puri) and Holland (1980/2010) radial wind field simulation.
4. **Vulnerability Functions**: CLIMADA/Eberenz damage curves mapping 1-min sustained wind speed to Mean Damage Ratio (MDR) by construction type.
5. **Financial Engine**: Ground-Up Loss (GUL), Deductibles, Policy Limits, and Gross Insured Loss.
6. **Sensitivity Analysis**: Testing how shifting storm track, intensity, and mitigation shifts portfolio losses.
7. **QGIS & Geospatial Mapping**: Cartographic layers and spatial visualizer.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

# Set project paths
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
DB_PATH = DATA_DIR / "exposure.db"
print(f"Project Root: {PROJECT_ROOT}")

## 1. SQL Scrubbing & Exposure Portfolio Inspection
Let's inspect the scrubbed exposure database and check the audit log for data hygiene interventions.

In [ ]:
conn = sqlite3.connect(DB_PATH)
# Check audit actions
df_audit = pd.read_sql_query("SELECT rule_name, action_taken, COUNT(*) as count FROM scrubbing_audit_log GROUP BY rule_name, action_taken", conn)
display(df_audit)

# Load clean exposure portfolio
df_exposure = pd.read_sql_query("SELECT * FROM scrubbed_exposure", conn)
conn.close()
print(f"Total Clean Locations: {len(df_exposure)}")
print(f"Total Insured Value (TIV): ₹{df_exposure['total_tiv'].sum() / 1e7:.2f} Crores")
df_exposure.head(5)

## 2. Hazard Track: NOAA IBTrACS Cyclone Fani (May 2019)
Cyclone Fani made landfall near Puri, Odisha on May 3, 2019 with sustained winds of 115 knots (~215 km/h).

In [ ]:
track_path = DATA_DIR / "hazard" / "fani_ibtracs_track.csv"
df_track = pd.read_csv(track_path)
df_track

## 3. Financial Loss Summaries by District and Construction Type

In [ ]:
df_dist_loss = pd.read_csv(RESULTS_DIR / "loss_summary_by_district.csv")
df_const_loss = pd.read_csv(RESULTS_DIR / "loss_summary_by_construction.csv")

print("=== LOSS BY DISTRICT ===")
display(df_dist_loss)

print("\n=== LOSS BY CONSTRUCTION TYPE ===")
display(df_const_loss)

## 4. Sensitivity Analysis: Testing Input Shifts
Catastrophe models are highly sensitive to small shifts in storm parameters. Let's inspect the sensitivity results.

In [ ]:
df_ts = pd.read_csv(RESULTS_DIR / "sensitivity_track_shift.csv")
df_is = pd.read_csv(RESULTS_DIR / "sensitivity_intensity_scaling.csv")
df_vm = pd.read_csv(RESULTS_DIR / "sensitivity_vulnerability_mitigation.csv")
df_ded = pd.read_csv(RESULTS_DIR / "sensitivity_deductibles.csv")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Track Shift
axes[0, 0].plot(df_ts["parameter_value"], df_ts["ground_up_loss_crores"], marker="o", color="darkred", lw=2)
axes[0, 0].axvline(0, color="gray", linestyle="--", label="Baseline Landfall (Puri)")
axes[0, 0].set_title("Sensitivity: Track Shift (km East/West) vs Loss", fontweight="bold")
axes[0, 0].set_ylabel("Ground-Up Loss (₹ Crores)")
axes[0, 0].legend()

# Intensity Scaling
axes[0, 1].bar(df_is["scenario_label"], df_is["ground_up_loss_crores"], color="navy")
axes[0, 1].set_title("Sensitivity: Storm Intensity Scaling (ΔP & Vmax)", fontweight="bold")
axes[0, 1].set_ylabel("Ground-Up Loss (₹ Crores)")

# Vulnerability Mitigation
axes[1, 0].plot(df_vm["parameter_value"] * 100, df_vm["ground_up_loss_crores"], marker="s", color="green", lw=2)
axes[1, 0].set_title("Sensitivity: Proactive Building Retrofitting", fontweight="bold")
axes[1, 0].set_xlabel("% Portfolio Retrofitted")
axes[1, 0].set_ylabel("Ground-Up Loss (₹ Crores)")

# Deductible Structure
axes[1, 1].bar(df_ded["deductible_rate_pct"], df_ded["gross_insured_loss_crores"], label="Insurer Gross Loss", color="royalblue")
axes[1, 1].bar(df_ded["deductible_rate_pct"], df_ded["retained_by_policyholder_crores"], bottom=df_ded["gross_insured_loss_crores"], label="Policyholder Retained", color="salmon")
axes[1, 1].set_title("Sensitivity: Policy Deductible Loss Absorption", fontweight="bold")
axes[1, 1].set_ylabel("Loss (₹ Crores)")
axes[1, 1].legend()

plt.tight_layout()
plt.show()